# Olist AI Analyst Copilot — SQL Analysis

## Day 1 + Day 2

Dataset: Olist Brazilian e-commerce dataset  
Database: DuckDB  
Purpose: SQL revision, business analysis, and portfolio documentation.

> **Important:** Monetary values are Brazilian reais (R$), not dollars.


In [1]:
import duckdb

con = duckdb.connect('../olist.duckdb')
con.sql('SHOW TABLES')


nameVARCHAR
clean_customers
clean_geolocation
clean_order_items
clean_order_payments
clean_order_reviews
clean_orders
clean_product_category_name_translation
clean_products
clean_sellers
raw_customers


# DAY 1 — Data exploration and core business metrics


## 1. Tables and row counts

**Business purpose:** Understand what data is available before analysis.


In [2]:
con.sql("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name;
""")


table_nameVARCHAR
clean_customers
clean_geolocation
clean_order_items
clean_order_payments
clean_order_reviews
clean_orders
clean_product_category_name_translation
clean_products
clean_sellers
raw_customers


In [3]:
con.sql("""
SELECT 'raw_orders' AS table_name, COUNT(*) AS row_count FROM raw_orders
UNION ALL SELECT 'raw_order_items', COUNT(*) FROM raw_order_items
UNION ALL SELECT 'raw_order_payments', COUNT(*) FROM raw_order_payments
UNION ALL SELECT 'raw_customers', COUNT(*) FROM raw_customers
UNION ALL SELECT 'raw_products', COUNT(*) FROM raw_products
UNION ALL SELECT 'raw_sellers', COUNT(*) FROM raw_sellers
UNION ALL SELECT 'raw_order_reviews', COUNT(*) FROM raw_order_reviews
UNION ALL SELECT 'raw_geolocation', COUNT(*) FROM raw_geolocation
UNION ALL SELECT 'raw_product_category_name_translation', COUNT(*) FROM raw_product_category_name_translation
ORDER BY table_name;
""")


table_nameVARCHAR,row_countBIGINT
raw_customers,99441
raw_geolocation,1000163
raw_order_items,112650
raw_order_payments,103886
raw_order_reviews,99224
raw_orders,99441
raw_product_category_name_translation,71
raw_products,32951
raw_sellers,3095


## 2. Verify `raw_orders` grain

**Question:** Is there exactly one row per order?

**Required verification:** `COUNT(*) = COUNT(DISTINCT order_id)`.


In [4]:
con.sql("""
SELECT COUNT(*) AS rows, COUNT(DISTINCT order_id) AS distinct_orders
FROM raw_orders;
""")


rowsBIGINT,distinct_ordersBIGINT
99441,99441


**Verified:** 99,441 rows = 99,441 distinct orders, so `raw_orders` is exactly 1 row per order.


## 3. Order status breakdown

**Grain:** 1 row per order. Not every status represents a completed sale.


In [5]:
con.sql("""
SELECT order_status, COUNT(*) AS orders,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_orders
FROM raw_orders
GROUP BY order_status
ORDER BY orders DESC;
""")


order_statusVARCHAR,ordersBIGINT,pct_ordersDOUBLE
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.3
created,5,0.01
approved,2,0.0


Known counts: delivered 96,478; shipped 1,107; canceled 625; unavailable 609; invoiced 314; processing 301; created 5; approved 2. Delivered share = **97.02%**.


## 4. Customer identity and repeat-customer rate


In [6]:
con.sql("""
SELECT COUNT(DISTINCT customer_id) AS customer_ids,
       COUNT(DISTINCT customer_unique_id) AS unique_customers
FROM raw_customers;
""")


customer_idsBIGINT,unique_customersBIGINT
99441,96096


In [7]:
con.sql("""
WITH customer_orders AS (
    SELECT customer_unique_id, COUNT(*) AS order_count
    FROM raw_customers
    GROUP BY customer_unique_id
)
SELECT COUNT(*) FILTER (WHERE order_count > 1) AS repeat_customers,
       COUNT(*) AS unique_customers,
       ROUND(COUNT(*) FILTER (WHERE order_count > 1) * 100.0 / COUNT(*), 2) AS repeat_rate_pct
FROM customer_orders;
""")


repeat_customersBIGINT,unique_customersBIGINT,repeat_rate_pctDOUBLE
2997,96096,3.12


**Known result:** 2,997 repeat customers / 96,096 unique customers = **3.12%**. This calculation uses all customer order records without an order-status filter; state that denominator/filter if used in final documentation.


## 5. Order date range


In [8]:
con.sql("""
SELECT MIN(order_purchase_timestamp) AS first_order,
       MAX(order_purchase_timestamp) AS last_order
FROM raw_orders;
""")


first_orderTIMESTAMP,last_orderTIMESTAMP
2016-09-04 21:15:19,2018-10-17 17:30:18


Known range: **2016-09-04 21:15:19 to 2018-10-17 17:30:18**.


## 6. Monthly delivered orders


In [9]:
con.sql("""
SELECT DATE_TRUNC('month', order_purchase_timestamp) AS month,
       COUNT(*) AS orders
FROM raw_orders
WHERE order_status = 'delivered'
GROUP BY month
ORDER BY month;
""")


monthTIMESTAMP,ordersBIGINT
2016-09-01 00:00:00,1
2016-10-01 00:00:00,265
2016-12-01 00:00:00,1
2017-01-01 00:00:00,750
2017-02-01 00:00:00,1653
2017-03-01 00:00:00,2546
2017-04-01 00:00:00,2303
2017-05-01 00:00:00,3546
2017-06-01 00:00:00,3135
2017-07-01 00:00:00,3872


Note: the raw dataset extends into October 2018, but the last month with delivered orders in this analysis is August 2018.


## 7. Category sales


In [10]:
con.sql("""
SELECT p.product_category_name AS category,
       ROUND(SUM(i.price), 2) AS sales
FROM raw_orders o
JOIN raw_order_items i ON o.order_id = i.order_id
JOIN raw_products p ON i.product_id = p.product_id
WHERE o.order_status = 'delivered'
GROUP BY category
ORDER BY sales DESC;
""")


categoryVARCHAR,salesDOUBLE
beleza_saude,1233131.72
relogios_presentes,1166176.98
cama_mesa_banho,1023434.76
esporte_lazer,954852.55
informatica_acessorios,888724.61
moveis_decoracao,711927.69
utilidades_domesticas,615628.69
cool_stuff,610204.1
automotivo,578966.65
brinquedos,471286.48


Known finding: `health_beauty` is the top category by item sales at roughly **R$1.23M**. Item volume and sales value can tell different stories.


## 8. Sales by customer state


In [11]:
con.sql("""
SELECT c.customer_state,
       ROUND(SUM(i.price), 2) AS sales
FROM raw_orders o
JOIN raw_customers c ON o.customer_id = c.customer_id
JOIN raw_order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY sales DESC;
""")


customer_stateVARCHAR,salesDOUBLE
SP,5067633.16
RJ,1759651.13
MG,1552481.83
RS,728897.47
PR,666063.51
SC,507012.13
BA,493584.14
DF,296498.41
GO,282836.7
ES,268643.45


## 9. Average delivery time by state


In [12]:
con.sql("""
SELECT c.customer_state,
       ROUND(AVG(DATE_DIFF('day', o.order_purchase_timestamp, o.order_delivered_customer_date)), 2) AS avg_delivery_days
FROM raw_orders o
JOIN raw_customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY avg_delivery_days DESC;
""")


customer_stateVARCHAR,avg_delivery_daysDOUBLE
RR,29.34
AP,27.18
AM,26.36
AL,24.5
PA,23.73
MA,21.51
SE,21.46
CE,21.2
AC,21.0
PB,20.39


Known example: RR ≈ **29.34 days**, SP ≈ **8.7 days**, showing substantial geographic differences.


## 10. Late-delivery rate


In [13]:
con.sql("""
WITH delivered AS (
    SELECT order_id, order_delivered_customer_date, order_estimated_delivery_date
    FROM raw_orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
)
SELECT COUNT(*) FILTER (WHERE order_delivered_customer_date > order_estimated_delivery_date) AS late_orders,
       COUNT(*) AS eligible_delivered_orders,
       ROUND(COUNT(*) FILTER (WHERE order_delivered_customer_date > order_estimated_delivery_date) * 100.0 / COUNT(*), 2) AS late_rate_pct
FROM delivered;
""")


late_ordersBIGINT,eligible_delivered_ordersBIGINT,late_rate_pctDOUBLE
7826,96470,8.11


Known result: **7,826 late / 96,470 eligible delivered orders = 8.11%**. Denominator = delivered orders with both actual and estimated delivery dates available.


## 11. Top 5 sellers by delivered item sales


In [14]:
con.sql("""
SELECT i.seller_id, ROUND(SUM(i.price), 2) AS sales
FROM raw_orders o
JOIN raw_order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
GROUP BY i.seller_id
ORDER BY sales DESC
LIMIT 5;
""")


seller_idVARCHAR,salesDOUBLE
4869f7a5dfa277a7dca6462dcf3b52b2,226987.93
53243585a1d6dc2643021fd1853d8905,217940.44
4a3ca9315b744ce9f8e9374361493884,196882.12
fa1c13f2614d7b5c4749cbc52fecda94,190917.14
7c67e1448b00f6e969d365cea6b010ab,186570.05


## 12. Correct payment total


In [15]:
con.sql("""
SELECT ROUND(SUM(payment_value), 2) AS total_payment_value
FROM raw_order_payments;
""")


total_payment_valueDOUBLE
16008872.12


Known result: **R$16,008,872.12**. Calculate this directly from the payment table rather than joining it to item rows.


## 13. Double-counting demonstration


In [16]:
con.sql("""
SELECT ROUND(SUM(p.payment_value), 2) AS inflated_payment_value
FROM raw_orders o
JOIN raw_order_items i ON o.order_id = i.order_id
JOIN raw_order_payments p ON o.order_id = p.order_id;
""")


inflated_payment_valueDOUBLE
20308134.71


Known bad-join result: **R$20,308,134.71**; inflation = **R$4,299,262.59**. An order with 3 items and 2 payments becomes 6 joined rows. Aggregate one side to the needed grain before joining; `DISTINCT` is not a generic fix.


# DAY 2 — Window functions and analytical SQL


## Window-function glossary

- `PARTITION BY` — splits rows into independent groups without collapsing them.
- `ROW_NUMBER()` — unique sequence within a partition.
- `RANK()` — ties share rank and later ranks can skip numbers.
- `LAG()` — previous available row.
- `LEAD()` — following row.
- `SUM() OVER` — windowed aggregate without collapsing rows.

**Important:** `LAG()` means previous available row, not automatically previous calendar month.


## 14. Top 3 categories by sales per state

**Grain:** state + category before ranking. **Filter:** delivered orders. **Metric:** `SUM(i.price)`.


In [17]:
con.sql("""
WITH category_sales AS (
    SELECT c.customer_state, p.product_category_name, ROUND(SUM(i.price), 2) AS sales
    FROM raw_orders o
    JOIN raw_customers c ON o.customer_id = c.customer_id
    JOIN raw_order_items i ON o.order_id = i.order_id
    JOIN raw_products p ON i.product_id = p.product_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state, p.product_category_name
), ranked AS (
    SELECT customer_state, product_category_name AS category, sales,
           RANK() OVER (PARTITION BY customer_state ORDER BY sales DESC) AS category_rank
    FROM category_sales
)
SELECT customer_state, category, sales, category_rank
FROM ranked
WHERE category_rank <= 3
ORDER BY customer_state, category_rank;
""")


customer_stateVARCHAR,categoryVARCHAR,salesDOUBLE,category_rankBIGINT
AC,esporte_lazer,1677.46,1
AC,relogios_presentes,1389.6,2
AC,beleza_saude,1386.58,3
AL,beleza_saude,12681.26,1
AL,relogios_presentes,11605.02,2
AL,informatica_acessorios,7605.61,3
AM,beleza_saude,2776.03,1
AM,ferramentas_jardim,1980.88,2
AM,informatica_acessorios,1815.74,3
AP,informatica_acessorios,2050.03,1


Expected observed result: 81 rows = 27 states × 3 ranks. Example SP: cama_mesa_banho R$472,238.07; beleza_saude R$453,916.48; relogios_presentes R$421,546.60. `RANK()` can return more than three rows on ties; use `ROW_NUMBER()` for exactly three rows per state.


## 15. Monthly GMV + MoM growth

Current notebook exercise: delivered orders; GMV = item price + freight value.

**Documentation warning:** the planned project glossary defines GMV as `SUM(price + freight_value)` for non-canceled orders. Keep the final project definition explicit and do not silently mix populations.


In [18]:
con.sql("""
WITH monthly AS (
    SELECT DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
           ROUND(SUM(i.price + i.freight_value), 2) AS gmv
    FROM raw_orders o
    JOIN raw_order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY month
), calendar AS (
    SELECT month FROM generate_series(
        (SELECT MIN(month) FROM monthly),
        (SELECT MAX(month) FROM monthly),
        INTERVAL '1 month'
    ) AS t(month)
), filled AS (
    SELECT c.month, COALESCE(m.gmv, 0) AS gmv
    FROM calendar c LEFT JOIN monthly m ON c.month = m.month
), monthly_lag AS (
    SELECT month, gmv, LAG(gmv) OVER (ORDER BY month) AS previous_gmv
    FROM filled
)
SELECT month, gmv, previous_gmv,
       ROUND((gmv - previous_gmv) / NULLIF(previous_gmv, 0) * 100, 2) AS mom_growth_pct
FROM monthly_lag
ORDER BY month;
""")


monthTIMESTAMP,gmvDOUBLE,previous_gmvDOUBLE,mom_growth_pctDOUBLE
2016-09-01 00:00:00,143.46,NULL,NULL
2016-10-01 00:00:00,46490.66,143.46,32306.71
2016-11-01 00:00:00,0.0,46490.66,-100.0
2016-12-01 00:00:00,19.62,0.0,NULL
2017-01-01 00:00:00,127482.37,19.62,649657.24
2017-02-01 00:00:00,271239.32,127482.37,112.77
2017-03-01 00:00:00,414330.95,271239.32,52.75
2017-04-01 00:00:00,390812.4,414330.95,-5.68
2017-05-01 00:00:00,566851.4,390812.4,45.04
2017-06-01 00:00:00,490050.37,566851.4,-13.55


Why the calendar CTE matters: `generate_series` creates every month, the `LEFT JOIN` keeps empty months, and `COALESCE` turns missing GMV into zero. This makes MoM compare adjacent calendar months rather than adjacent rows that happen to exist.


## 16. First-to-second-order gap


In [19]:
con.sql("""
WITH ranked_orders AS (
    SELECT c.customer_unique_id, o.order_purchase_timestamp,
           ROW_NUMBER() OVER (
               PARTITION BY c.customer_unique_id
               ORDER BY o.order_purchase_timestamp
           ) AS order_number
    FROM raw_orders o
    JOIN raw_customers c ON o.customer_id = c.customer_id
), date_orders AS (
    SELECT customer_unique_id,
           MAX(CASE WHEN order_number = 1 THEN order_purchase_timestamp END) AS first_order,
           MAX(CASE WHEN order_number = 2 THEN order_purchase_timestamp END) AS second_order
    FROM ranked_orders
    GROUP BY customer_unique_id
), order_gaps AS (
    SELECT customer_unique_id, first_order, second_order,
           DATE_DIFF('day', first_order, second_order) AS days_between
    FROM date_orders
)
SELECT AVG(days_between) AS avg_days_to_second_order
FROM order_gaps
WHERE second_order IS NOT NULL;
""")


avg_days_to_second_orderDOUBLE
80.34834834834835


Known result: **80.35 days**. One-order customers are excluded because they have no second-order date and therefore no valid gap. The explicit filter makes the population clear.


## 17. Monthly AOV


In [20]:
con.sql("""
WITH order_values AS (
    SELECT o.order_id,
           DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
           SUM(i.price + i.freight_value) AS order_value
    FROM raw_orders o
    JOIN raw_order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY o.order_id, month
)
SELECT month, ROUND(AVG(order_value), 2) AS aov
FROM order_values
GROUP BY month
ORDER BY month;
""")


monthTIMESTAMP,aovDOUBLE
2016-09-01 00:00:00,143.46
2016-10-01 00:00:00,175.44
2016-12-01 00:00:00,19.62
2017-01-01 00:00:00,169.98
2017-02-01 00:00:00,164.09
2017-03-01 00:00:00,162.74
2017-04-01 00:00:00,169.7
2017-05-01 00:00:00,159.86
2017-06-01 00:00:00,156.32
2017-07-01 00:00:00,146.25


AOV must be calculated from one row per order. Averaging item rows would give orders with more items more weight.


# Key findings

1. Repeat customer rate: **3.12%**.
2. Late delivery rate: **8.11%** among 96,470 eligible delivered orders.
3. Average first-to-second-order gap: **80.35 days**.
4. Delivered orders: **96,478 / 99,441 = 97.02%**.
5. Direct payment total: **R$16,008,872.12**.
6. Bad item-payment join: **R$20,308,134.71**, inflating by **R$4,299,262.59**.
7. Delivery time differs substantially by state.
8. Sales value and item volume can produce different category rankings.


# Day 3
1. Profile the 775 orders with no items
Business question: Are these genuinely invalid orders, or are they legitimate orders in a non-sale status?

In [21]:
con.sql("""
SELECT
    o.order_status,
    COUNT(*) AS orders_without_items
FROM raw_orders o
LEFT JOIN raw_order_items i
    ON o.order_id = i.order_id
WHERE i.order_id IS NULL
GROUP BY o.order_status
ORDER BY orders_without_items DESC;
""")

order_statusVARCHAR,orders_without_itemsBIGINT
unavailable,603
canceled,164
created,5
invoiced,2
shipped,1


# Check 2: delivered orders with a missing delivery date.
1. Business question
 Among orders marked delivered, are there any where the actual delivery date is missing?

In [22]:
con.sql("""
SELECT
    order_status,
    order_id,
    order_delivered_customer_date
FROM raw_orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NULL;
""")

order_statusVARCHAR,order_idVARCHAR,order_delivered_customer_dateTIMESTAMP
delivered,2d1e2d5bf4dc7227b3bfebb81328c15f,NULL
delivered,f5dd62b788049ad9fc0526e3ad11a097,NULL
delivered,2ebdfc4f15f23b91474edf87475f108e,NULL
delivered,e69f75a717d64fc5ecdfae42b2e8e086,NULL
delivered,0d3268bad9b086af767785e3f0fc0133,NULL
delivered,2d858f451373b04fb5c984a1cc2defaf,NULL
delivered,ab7c89dc1bf4a1ead9d6ec1ec8968a84,NULL
delivered,20edc82cf5400ce95e1afacc25798b31,NULL


In [23]:
con.sql("""
SELECT COUNT(*) AS affected_orders
FROM raw_orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NULL;
""")

affected_ordersBIGINT
8


# 3.Check impossible delivery dates

In [24]:
con.sql("""
SELECT COUNT(*) AS impossible_delivery_dates
FROM raw_orders
WHERE order_delivered_customer_date < order_purchase_timestamp
  AND order_status = 'delivered';
""")

impossible_delivery_datesBIGINT
0


# 4.Multiple reviews for the same order

In [25]:
con.sql("""
SELECT
    order_id,
    COUNT(*) AS review_count
FROM raw_order_reviews
GROUP BY order_id
HAVING COUNT(*) > 1;
""")

order_idVARCHAR,review_countBIGINT
b6e5aa946acc4e29e7069510f28a0bce,2
d07ea9203ae0a6ea15a2eacc87b4c0f7,2
3fe4dbcdb046a475dbf25463c1ca78bd,2
892e04325197686b3f5a32eb5fb38177,2
e73eb0a2c088eec6c479d66ac0d910f0,2
65709b9620e3ef6f575918b8348a8ece,2
246b14cc444e8a7df0afd80495dfa052,2
c7cfea0c153e6382e32e84c2a9dd7d2e,2
3e93213bb8fdda91186b4018b2fe0030,2
3c557fef5b5d9315a99f7442ef013a8e,2


In [26]:
con.sql("""
SELECT
    review_id,
    COUNT(*) AS review_count
FROM raw_order_reviews
GROUP BY review_id
HAVING COUNT(*) > 1;
""")

review_idVARCHAR,review_countBIGINT
5395e388be6105b6a76ddcce4557356f,2
27707bdab4a7b6bb49a2c3bcdf41e995,2
ddacc3ef27deefcbf5c3d7565a1e6270,2
f45e3373e177f56944a549ff601f838b,2
c7528903f66f3c8400f50ae8525e77f6,2
5a7142e97f58c992e5c19043f70ac1c5,2
28d0d8241ef547ebff745da0b395e7cf,2
be96880817b432a0cc5fd6083d521c10,2
db90ab0ea5d97cbfa75cb66ea71013cc,2
c1dd7c6a2154caaf1a6accd68842286f,2


In [27]:
con.sql("""
SELECT *
FROM raw_order_reviews
WHERE review_id = 'b06f6882b1c0bc82ddf057b58b86fe50';
""")

review_idVARCHAR,order_idVARCHAR,review_scoreBIGINT,review_comment_titleVARCHAR,review_comment_messageVARCHAR,review_creation_dateTIMESTAMP,review_answer_timestampTIMESTAMP
b06f6882b1c0bc82ddf057b58b86fe50,2daee070f2042c8b7a8e9fdde778a31a,5,Muito bom,"Ótimo, excelente produto recomendo a todos",2018-07-12 00:00:00,2018-07-13 10:49:12
b06f6882b1c0bc82ddf057b58b86fe50,1536003b9af6a526e0d02b8cd13fd2af,5,Muito bom,"Ótimo, excelente produto recomendo a todos",2018-07-12 00:00:00,2018-07-13 10:49:12


In [28]:
con.sql("""
SELECT
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp,
    COUNT(*) AS duplicate_count
FROM raw_order_reviews
GROUP BY
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
HAVING COUNT(*) > 1;
""")

review_idVARCHAR,order_idVARCHAR,review_scoreBIGINT,review_comment_titleVARCHAR,review_comment_messageVARCHAR,review_creation_dateTIMESTAMP,review_answer_timestampTIMESTAMP,duplicate_countBIGINT


# 5.check where payment value is zero

In [29]:
con.sql("""
SELECT
    COUNT(*) AS zero_value_payments
FROM raw_order_payments
WHERE payment_value = 0;
""")

zero_value_paymentsBIGINT
9


In [30]:
con.sql("""
SELECT *
FROM raw_order_payments
WHERE payment_value = 0;
""")

order_idVARCHAR,payment_sequentialBIGINT,payment_typeVARCHAR,payment_installmentsBIGINT,payment_valueDOUBLE
8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [31]:
con.sql("""
SELECT
    payment_type,
    COUNT(*) AS payment_rows,
    SUM(CASE WHEN payment_value = 0 THEN 1 ELSE 0 END) AS zero_value_rows
FROM raw_order_payments
GROUP BY payment_type
ORDER BY payment_rows DESC;
""")

payment_typeVARCHAR,payment_rowsBIGINT,zero_value_rowsHUGEINT
credit_card,76795,0
boleto,19784,0
voucher,5775,6
debit_card,1529,0
not_defined,3,3


In [32]:
con.sql("""
SELECT *
FROM raw_order_payments
WHERE payment_type = 'not_defined';
""")

order_idVARCHAR,payment_sequentialBIGINT,payment_typeVARCHAR,payment_installmentsBIGINT,payment_valueDOUBLE
4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [33]:
con.sql("""
SELECT
    order_id,
    COUNT(*) AS payment_rows,
    SUM(payment_value) AS total_payment
FROM raw_order_payments
WHERE order_id IN (
    SELECT order_id
    FROM raw_order_payments
    WHERE payment_type = 'not_defined'
)
GROUP BY order_id;
""")

order_idVARCHAR,payment_rowsBIGINT,total_paymentDOUBLE
4637ca194b6387e2d538dc89b124b0ee,1,0.0
c8c528189310eaa44a745b8d9d26908b,1,0.0
00b1cb0320190ca0daa2c88b35206009,1,0.0


In [34]:
con.sql("""
DESCRIBE raw_products;
""")

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
product_id,VARCHAR,YES,NULL,NULL,NULL
product_category_name,VARCHAR,YES,NULL,NULL,NULL
product_name_lenght,BIGINT,YES,NULL,NULL,NULL
product_description_lenght,BIGINT,YES,NULL,NULL,NULL
product_photos_qty,BIGINT,YES,NULL,NULL,NULL
product_weight_g,BIGINT,YES,NULL,NULL,NULL
product_length_cm,BIGINT,YES,NULL,NULL,NULL
product_height_cm,BIGINT,YES,NULL,NULL,NULL
product_width_cm,BIGINT,YES,NULL,NULL,NULL


In [35]:
con.sql("""
SELECT
    COUNT(*) AS null_category_products
FROM raw_products
WHERE product_category_name IS NULL;
""")

null_category_productsBIGINT
610


In [36]:
con.sql("""
SELECT COUNT(*) AS blank_category_products
FROM raw_products
WHERE product_category_name = '';
""")

blank_category_productsBIGINT
0


# Geolocation profile

In [37]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT geolocation_zip_code_prefix) AS unique_zip_prefixes
FROM raw_geolocation;
""")

total_rowsBIGINT,unique_zip_prefixesBIGINT
1000163,19015


In [38]:
con.sql("""
SELECT
    COUNT(*) AS zip_prefixes_with_multiple_rows
FROM (
    SELECT geolocation_zip_code_prefix
    FROM raw_geolocation
    GROUP BY geolocation_zip_code_prefix
    HAVING COUNT(*) > 1
);
""")

zip_prefixes_with_multiple_rowsBIGINT
17972


In [39]:
con.sql("""
SELECT
    order_status,
    COUNT(*) AS orders_without_items
FROM raw_orders o
LEFT JOIN raw_order_items i
    ON o.order_id = i.order_id
WHERE i.order_id IS NULL
GROUP BY order_status
ORDER BY orders_without_items DESC;
""")

order_statusVARCHAR,orders_without_itemsBIGINT
unavailable,603
canceled,164
created,5
invoiced,2
shipped,1


In [40]:
con.sql("""
SELECT
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp
FROM raw_orders o
LEFT JOIN raw_order_items i
    ON o.order_id = i.order_id
WHERE i.order_id IS NULL
  AND o.order_status IN ('shipped', 'invoiced');
""")

order_idVARCHAR,order_statusVARCHAR,order_purchase_timestampTIMESTAMP
2ce9683175cdab7d1c95bcbb3e36f478,invoiced,2016-10-05 21:03:33
e04f1da1f48bf2bbffcf57b9824f76e1,invoiced,2016-10-05 13:22:20
a68ce1686d536ca72bd2dadc4b8671e5,shipped,2016-10-05 01:47:40


In [41]:
con.sql("""
SELECT
    order_id,
    payment_type,
    payment_value,
    payment_sequential
FROM raw_order_payments
WHERE payment_type = 'not_defined'
  AND payment_value = 0;
""")

order_idVARCHAR,payment_typeVARCHAR,payment_valueDOUBLE,payment_sequentialBIGINT
4637ca194b6387e2d538dc89b124b0ee,not_defined,0.0,1
00b1cb0320190ca0daa2c88b35206009,not_defined,0.0,1
c8c528189310eaa44a745b8d9d26908b,not_defined,0.0,1
